## Domain Adaptation

Usually, you can fine-tune a pretrained model directly on your task data.

If your data uses specialized language, such as legal or scientific text, first continue training the model on that type of text. This helps it learn the domain’s vocabulary and writing style.

This step is called **domain adaptation**. Afterward, fine-tune the model for your specific task.

In [1]:
CHECKPOINT = "distilbert/distilbert-base-uncased"

Although BERT and RoBERTa models are widely used, this example uses **DistilBERT**, a smaller model that trains faster while often retaining similar performance on downstream tasks.

DistilBERT was created using **knowledge distillation**: a larger “teacher” model, such as BERT, guides the training of a smaller “student” model. The details of this technique are beyond this section.

In [3]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForMaskedLM.from_pretrained(CHECKPOINT)

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning

In [4]:
model.num_parameters()

66985530

In [5]:
text = 'This is a great [MASK]'

In [6]:
import torch

In [7]:
inputs = tokenizer(text, return_tensors='pt')
token_logits = model(**inputs).logits

In [11]:
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

In [12]:
for token in top_5_tokens:
    print(f"> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}")

> This is a great !
> This is a great .
> This is a great deal
> This is a great adventure
> This is a great ;


In [13]:
from datasets import load_dataset

In [17]:
imdb_dataset = load_dataset("stanfordnlp/imdb")
imdb_dataset

C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\huggingface_cache\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 564474.34 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [18]:
sample = imdb_dataset['train'].shuffle(seed=42).select(range(3))

In [19]:
for row in sample:
    print(f"\n'>>> Review: {row['text']}'")
    print(f"'>>> Label: {row['label']}'")


'>>> Review: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...'
'>>> Label: 1'

'>>> Review: This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stu

In [20]:
for split in ["train", "test", "unsupervised"]:
    print(split, set(imdb_dataset[split]["label"]))

train {0, 1}
test {0, 1}
unsupervised {-1}


In [21]:
def tokenize_function(examples):
    results = tokenizer(examples['text'])

    if tokenizer.is_fast:
        results['word_ids'] = [results.word_ids(i) for i in range(len(results['input_ids']))]

    return results


In [22]:
tokenized_dataset = imdb_dataset.map(tokenize_function, batched=True, remove_columns=['text', 'label'])

Map: 100%|██████████| 50000/50000 [00:12<00:00, 3865.66 examples/s]


In [28]:
chunk_size = 128

Note that using a small chunk size can be detrimental in real-world scenarios, so you should use a size that corresponds to the use case you will apply your model to.

In [29]:
tokenized_samples = tokenized_dataset["train"][:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> Review {idx} length: {len(sample)}'")

'>>> Review 0 length: 363'
'>>> Review 1 length: 304'
'>>> Review 2 length: 133'


In [30]:
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // chunk_size) * chunk_size
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }

    result["labels"] = result["input_ids"].copy()
    return result

In [31]:
lm_datasets = tokenized_dataset.map(group_texts, batched=True)
lm_datasets

Map: 100%|██████████| 50000/50000 [02:45<00:00, 302.61 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 61291
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 59904
    })
    unsupervised: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 122957
    })
})

5040